# DR Score training pipeline
Participant-grouped model development through knee-level DR Score generation. No post-hoc analyses are included.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
from drscore.config import load_config
from drscore.data import load_metadata
from drscore.training import run_cross_validation

config = load_config(ROOT / 'configs' / 'final_model.yaml')
metadata_path = ROOT / config['data']['metadata_csv']
data = load_metadata(metadata_path, allowed_cohorts=tuple(config['data']['allowed_cohorts']))
data.groupby('cohort').agg(knees=('knee_id', 'size'), participants=('participant_id', 'nunique'), events=('event', 'sum'))

## Primary analysis
The prespecified primary seed runs five participant-grouped outer folds.

In [ ]:
primary_seed = int(config['training']['primary_seed'])
selection = run_cross_validation(
    data,
    config,
    ROOT / 'outputs' / 'primary',
    seeds=[primary_seed],
    num_workers=0,
)
selection

## Optional seed-sensitivity repetitions

In [ ]:
RUN_SENSITIVITY_SEEDS = False
if RUN_SENSITIVITY_SEEDS:
    all_seeds = [primary_seed, *map(int, config['training']['sensitivity_seeds'])]
    sensitivity_selection = run_cross_validation(
        data, config, ROOT / 'outputs' / 'all_seeds', seeds=all_seeds, num_workers=0
    )
    display(sensitivity_selection)